# OmniPhyto: Drive → GitHub bridge (Colab)

Run the cells **in order**. Nothing is pushed until the last cell, and that cell also needs `PUSH = True`.

1. **Cell 1** connects Drive and clones your repo (read-only at this point).
2. **Cell 2: SCAN.** Reads the repo and prints a report. Copy the whole report and paste it back to Claude. It is also saved to Drive as `omniphyto_run/repo_scan.md`.
3. **Cell 3: PLAN.** Shows exactly which Drive files would be copied where, and builds the folder layout inside the local clone. Dry run by default.
4. **Cell 4: PUSH.** Commits the new layout and data and pushes it to the branch you choose.

**One-time setup:** make a GitHub *fine-grained personal access token* that has **Contents: Read and write** on this one repository. In Colab open the key icon (Secrets) in the left bar, add a secret named `GITHUB_TOKEN`, paste the token, and switch on "Notebook access". The token is never printed or saved in the notebook.

In [ ]:
# CELL 1: settings, Drive, clone
import os, sys, re, json, time, shutil, hashlib, subprocess, glob, textwrap
from pathlib import Path
from datetime import datetime

# ===================== SETTINGS =====================
GITHUB_REPO  = 'itsrex212-arch/Herba-Quest'   # your repository (owner/name)
BRANCH       = 'main'            # branch to read (and the branch the site deploys from)
PUSH_BRANCH  = 'omniphyto-data'  # branch Cell 4 pushes to. Use the same as BRANCH to publish straight to the live site
PUSH         = False             # Cell 4 only pushes when this is True
DRY_RUN      = True              # Cell 3 only lists the plan when this is True; set False to copy files into the clone
POSE_SCOPE   = 'all'             # 'all' = every pose file (best for the 3D viewer); 'top' = only compounds in the safety panel and top 50
INCLUDE_OVERLAYS = True          # also copy the redocking overlay PDBs for the validation page
INCLUDE_PREPARED = True          # prepared receptors (.pdbqt, box, config), the 238 ligand .pdbqt files and the redocking files
INCLUDE_PLIP     = True          # PLIP interaction complexes for the top compounds
INCLUDE_NOTEBOOKS = True         # copy the OmniPhyto notebooks from 'Colab Notebooks' (outputs removed)
MD_ALSO_ROOT = True              # put the MD report in md_report/ at the top of the repo too (the currently deployed app.py reads that folder)
MD_NOTES = {'OP0437': 'negative control: weak binder (Vina -4.2), not expected to hold a pose'}   # text added to the notes column of md_summary.csv
GIT_NAME     = 'OmniPhyto team'
GIT_EMAIL    = 'omniphyto@users.noreply.github.com'   # a placeholder so no personal address ends up in the commits
# =====================================================

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive'); IN_COLAB = True
except Exception as e:
    print('Not in Colab (or the Drive mount failed):', e)

def human(n):
    for u in ('B', 'KB', 'MB', 'GB'):
        if n < 1024 or u == 'GB': return f'{n:.0f} {u}' if u == 'B' else f'{n:.1f} {u}'
        n /= 1024

def find_project():
    for root in ('/content/drive/MyDrive', '/content/drive/Shareddrives'):
        if not os.path.isdir(root): continue
        base = root.rstrip('/').count('/')
        for dp, dn, fn in os.walk(root, followlinks=True):
            dn[:] = [d for d in dn if not d.startswith('.')]
            if dp.count('/') - base >= 4: dn[:] = []
            if 'omniphyto_run' in dn: return dp
    return ''

PROJECT = Path(os.environ.get('OMNI_PROJECT') or find_project() or '.')
RUN = PROJECT / 'omniphyto_run'
if not RUN.exists(): raise SystemExit(f'Could not find omniphyto_run/ under {PROJECT}. Check the Drive folder.')
CLONE = Path(os.environ.get('OMNI_CLONE', '/content/repo'))

TOKEN = ''
try:
    from google.colab import userdata
    TOKEN = (userdata.get('GITHUB_TOKEN') or '').strip()
except Exception as e:
    TOKEN = os.environ.get('GITHUB_TOKEN', '').strip()
    if not TOKEN: print('Could not read the Colab secret GITHUB_TOKEN:', type(e).__name__)
if not TOKEN and not os.environ.get('OMNI_REPO_URL'):
    raise SystemExit('No GitHub token found, and the repo is private.\n'
                     '1) Click the key icon (Secrets) in the left bar of Colab.\n'
                     '2) Add a secret named exactly GITHUB_TOKEN and paste the token as its value.\n'
                     '3) Switch ON "Notebook access" for it, then run this cell again.')
CLEAN_URL = os.environ.get('OMNI_REPO_URL') or f'https://github.com/{GITHUB_REPO}.git'
AUTH_URL = CLEAN_URL.replace('https://', f'https://x-access-token:{TOKEN}@') if (TOKEN and CLEAN_URL.startswith('https://')) else CLEAN_URL

def git(*args, cwd=None, url_arg=False, check=True):
    """run git; the token (if any) is hidden from any output"""
    r = subprocess.run(['git', *args], cwd=cwd or CLONE, capture_output=True, text=True, env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    out = (r.stdout + r.stderr).replace(TOKEN, '***') if TOKEN else (r.stdout + r.stderr)
    if check and r.returncode != 0: raise RuntimeError(f'git {args[0]} failed:\n{out.strip()[-600:]}')
    return out.strip()

if CLONE.exists(): shutil.rmtree(CLONE)
git('clone', '--depth', '50', '--branch', BRANCH, AUTH_URL, str(CLONE), cwd='.')
git('remote', 'set-url', 'origin', CLEAN_URL)                 # keep the token out of .git/config
git('config', 'user.name', GIT_NAME); git('config', 'user.email', GIT_EMAIL)
print('Drive project :', PROJECT)
print('Cloned        :', CLEAN_URL, '| branch', BRANCH, '| HEAD', git('rev-parse', '--short', 'HEAD'))

In [ ]:
# CELL 2: SCAN the repository (read-only). Copy the printed report back to Claude.
def scan_repo(root: Path) -> str:
    L = []; add = L.append
    tracked = [p for p in git('ls-files', cwd=root).splitlines() if p]
    sizes = {p: (root / p).stat().st_size for p in tracked if (root / p).is_file()}
    add('# OmniPhyto repo scan'); add(f'_{datetime.now():%Y-%m-%d %H:%M}_\n')
    add('## 1. Repository')
    add(f'- remote: {CLEAN_URL}'); add(f'- branch: {git("rev-parse", "--abbrev-ref", "HEAD", cwd=root)} | HEAD {git("rev-parse", "--short", "HEAD", cwd=root)}')
    add('- branches on remote: ' + ', '.join(sorted({b.strip().replace("origin/", "") for b in git("branch", "-r", cwd=root).splitlines() if "->" not in b})) )
    add('- last commits:')
    for l in git('log', '-8', '--pretty=%h | %ad | %an | %s', '--date=short', cwd=root).splitlines(): add(f'  - {l}')
    add(f'- tracked files: {len(tracked)} | total size {human(sum(sizes.values()))}')
    add('- biggest tracked files: ' + '; '.join(f'{p} ({human(s)})' for p, s in sorted(sizes.items(), key=lambda x: -x[1])[:8]))
    add('\n## 2. Folder tree (tracked files)')
    tree = {}
    for p in tracked:
        parts = p.split('/')
        d = tree
        for part in parts[:-1]: d = d.setdefault(part + '/', {})
        d[parts[-1]] = sizes.get(p, 0)
    def show(d, depth=0, maxd=3):
        pad = '  ' * depth
        files = sorted(k for k in d if not k.endswith('/')); dirs = sorted(k for k in d if k.endswith('/'))
        for k in dirs:
            n = sum(1 for _ in walk(d[k]))
            add(f'{pad}- {k} ({n} files)')
            if depth + 1 < maxd: show(d[k], depth + 1, maxd)
        if len(files) <= 12:
            for k in files: add(f'{pad}- {k} ({human(d[k])})')
        else:
            for k in files[:6]: add(f'{pad}- {k} ({human(d[k])})')
            add(f'{pad}- ... and {len(files) - 6} more files here')
    def walk(d):
        for k, v in d.items():
            if isinstance(v, dict): yield from walk(v)
            else: yield k
    show(tree)

    add('\n## 3. Target layout check')
    expect = ['app.py', 'requirements.txt', '.gitignore', 'README.md', 'omniphyto/__init__.py', 'omniphyto_contract/schemas.py',
              'omniphyto_contract/check_data.py', 'pages/', 'engines/__init__.py', 'assets/', 'data/incoming/', 'data/receptors/',
              'data/poses/', 'data/md_report/', 'data/incoming/target_card.json', '.streamlit/config.toml']
    for e in expect:
        ok = any(p == e or p.startswith(e) for p in tracked) if not e.endswith('/') else any(p.startswith(e) for p in tracked)
        add(f'- [{"x" if ok else " "}] {e}')

    add('\n## 4. app.py')
    ap = root / 'app.py'
    if ap.exists():
        t = ap.read_text(encoding='utf-8', errors='replace'); lines = t.splitlines()
        add(f'- {len(lines)} lines, {human(ap.stat().st_size)}')
        mods = sorted({m.group(1).split('.')[0] for m in re.finditer(r'^\s*(?:from|import)\s+([A-Za-z_][\w\.]*)', t, re.M)})
        add('- imports: ' + ', '.join(mods))
        add('- local package imports: ' + (', '.join(sorted({m.group(0).strip() for m in re.finditer(r'^from omniphyto[\w\.]* import .*$', t, re.M)})) or 'none'))
        m = re.search(r'^PAGES\s*=\s*\[(.*?)\]', t, re.M | re.S)
        add('- pages list: ' + (re.sub(r'\s+', ' ', m.group(1))[:300] if m else 'not found (maybe st.navigation / pages/)'))
        add('- path constants: ' + '; '.join(sorted({l.strip()[:90] for l in lines if re.match(r'^[A-Z_]+\s*=\s*ROOT', l)})))
        big = max((len(l) for l in lines), default=0)
        add(f'- longest line: {human(big)} (a very long line usually means an embedded base64 image)')
        add(f'- file_uploader calls: {t.count("st.file_uploader")} | FAKE_SAMPLE mentions: {t.count("FAKE_SAMPLE")} | set_page_config: {"yes" if "set_page_config" in t else "no"} | st.navigation: {"yes" if "st.navigation" in t else "no"}')
    else: add('- app.py NOT found at the repo root')

    add('\n## 5. Python package folder `omniphyto/`')
    pk = [p for p in tracked if p.startswith('omniphyto/')]
    add('- ' + (', '.join(f'{p.split("/", 1)[1]} ({human(sizes[p])})' for p in pk) if pk else 'none'))

    add('\n## 6. Dependencies and hosting files')
    for f in ('requirements.txt', 'packages.txt', 'runtime.txt', '.python-version', 'Procfile', 'Dockerfile', 'render.yaml', '.streamlit/config.toml', '.gitattributes'):
        p = root / f
        if p.exists():
            add(f'- {f}:'); add('  ```'); add(textwrap.indent(p.read_text(errors="replace").strip()[:700], '  ')); add('  ```')
        else: add(f'- {f}: not present')
    wf = [p for p in tracked if p.startswith('.github/workflows/')]
    add('- GitHub Actions: ' + (', '.join(wf) if wf else 'none'))

    add('\n## 7. Existing data files')
    dfiles = [p for p in tracked if p.startswith('data/')]
    add(f'- {len(dfiles)} tracked files under data/')
    for p in dfiles[:25]: add(f'  - {p} ({human(sizes[p])})')
    if len(dfiles) > 25: add(f'  - ... and {len(dfiles) - 25} more')
    fake = []
    for p in dfiles:
        if p.endswith('.csv') and sizes[p] < 5e6 and 'FAKE_SAMPLE' in (root / p).read_text(errors='replace'): fake.append(p)
    add('- CSV files that contain FAKE_SAMPLE rows: ' + (', '.join(fake) if fake else 'none'))

    add('\n## 8. Safety checks')
    pats = {'GitHub token': r'gh[pousr]_[A-Za-z0-9]{30,}|github_pat_[A-Za-z0-9_]{30,}', 'AWS key': r'AKIA[0-9A-Z]{16}',
            'private key': r'-----BEGIN [A-Z ]*PRIVATE KEY-----', 'API key': r'sk-[A-Za-z0-9_\-]{30,}'}
    hits = []
    for p in tracked:
        if sizes.get(p, 0) > 2e6 or p.endswith(('.png', '.jpg', '.jpeg', '.gif', '.pdf', '.zip')): continue
        try: txt = (root / p).read_text(errors='ignore')
        except Exception: continue
        for name, rx in pats.items():
            if re.search(rx, txt): hits.append(f'{p} ({name})')
    add('- possible secrets in tracked files (values not shown): ' + ('; '.join(hits) if hits else 'none found'))
    add('- files over 50 MB: ' + (', '.join(p for p, s in sizes.items() if s > 50e6) or 'none'))
    add('- .gitignore present: ' + ('yes' if (root / '.gitignore').exists() else 'NO'))
    return '\n'.join(L)

REPORT = scan_repo(CLONE)
print(REPORT)
(RUN / 'repo_scan.md').write_text(REPORT, encoding='utf-8')
print('\n--- saved to Drive:', RUN / 'repo_scan.md', '---')

In [ ]:
# CELL 3: PLAN the Drive -> repo copy (dry run by default), and build the layout in the local clone
import csv as _csv
import pandas as pd

DEST_INCOMING = CLONE / 'data' / 'incoming'
plan = []        # (source Path or bytes, dest Path, note)

def newest(patterns, exclude=()):
    hits = []
    for pat in patterns:
        for base in (PROJECT, RUN, RUN / 'results'):
            hits += [Path(x) for x in glob.glob(str(base / pat))]
    hits = [h for h in hits if h.is_file() and not any(e in h.name for e in exclude)]
    return max(hits, key=lambda h: h.stat().st_mtime) if hits else None

def add_file(src, dest_rel, note=''):
    if src is None or not Path(src).exists(): plan.append((None, dest_rel, 'MISSING in Drive ' + note)); return
    plan.append((Path(src), dest_rel, note))

# --- result tables from the docking notebook ---
for name in ('docking_target.csv', 'docking_antitargets.csv', 'summary_report.csv', 'ligand_prep_log.csv'):
    add_file(RUN / 'results' / name, f'data/incoming/{name}')

# --- redocking table, converted to the column names the app expects ---
rs = RUN / 'validation' / 'redock_summary.csv'
redock_bytes = None
if rs.exists():
    r = pd.read_csv(rs)
    nice = {'dpp4': 'DPP-4', 'herg': 'hERG', 'cyp3a4': 'CYP3A4', 'cyp2d6': 'CYP2D6'}
    out = pd.DataFrame({'protein': r.receptor.map(lambda x: nice.get(x, x)), 'status': r.verdict.astype(str).str.split(' ').str[0].str.upper(),
                        'ligand': r.get('ligand'), 'best_vina_score': r.get('ref_score'), 'rmsd_top_pose': r.get('top1_rmsd'),
                        'rmsd_best_of_9': r.get('best_rmsd'), 'reference_score_kcal': r.get('ref_score')})
    redock_bytes = out.to_csv(index=False).encode()
    plan.append((redock_bytes, 'data/incoming/redocking_results.csv', 'converted from validation/redock_summary.csv'))
else: plan.append((None, 'data/incoming/redocking_results.csv', 'MISSING in Drive (validation/redock_summary.csv)'))

# --- ML and master tables, target card, final ranking (newest match wins) ---
for label, pats, excl in (('ML detail table', ['*scored_detail*.csv'], ()), ('ML compact table', ['*scored.csv'], ('detail',)),
                          ('master compound table', ['*compounds_with_DOIs*.csv', '*compound_structures*.csv'], ()),
                          ('final ranking', ['ranking*.csv'], ()), ('target card', ['*target_card*.json'], ())):
    f = newest(pats, excl)
    dn = 'target_card.json' if label == 'target card' else (f.name if f else f'<{label}>')      # the app looks for exactly target_card.json
    add_file(f, f'data/incoming/{dn}', f'({label})')

# --- poses ---
wanted = None
if POSE_SCOPE == 'top':
    keep = set()
    for nm in ('docking_antitargets.csv', 'summary_report.csv'):
        if (RUN / 'results' / nm).exists(): keep |= set(pd.read_csv(RUN / 'results' / nm).compound_id.astype(str))
    t = pd.read_csv(RUN / 'results' / 'docking_target.csv'); keep |= set(t.sort_values('vina_score_kcal').head(50).compound_id.astype(str)); wanted = keep
n_pose = 0
for sub in ('target', 'herg', 'cyp3a4', 'cyp2d6'):
    for f in sorted((RUN / 'results' / sub / 'poses').glob('*_out.pdbqt')):
        if wanted is not None and f.name.replace('_out.pdbqt', '') not in wanted: continue
        plan.append((f, f'data/incoming/results/{sub}/poses/{f.name}', 'pose')); n_pose += 1

# --- raw receptor structures for the 3D viewer ---
for pid in ('3D4L', '8ZYO', '2V0M', '3TDA'): add_file(RUN / 'data_raw' / f'{pid}.pdb', f'data/receptors/{pid}.pdb', '(receptor)')

# --- MD report (from the MD notebook; appears only after the simulation has been analysed) ---
for f in sorted((RUN / 'md_report').glob('*')) if (RUN / 'md_report').is_dir() else []:
    if not (f.is_file() and f.suffix in ('.csv', '.png')): continue
    for dd in ['data/incoming/md_report'] + (['md_report'] if MD_ALSO_ROOT else []):
        if f.name == 'md_summary.csv' and MD_NOTES:
            _m = pd.read_csv(f)
            if 'notes' not in _m.columns: _m['notes'] = ''                      # file without a notes column: add one
            _m['notes'] = _m['notes'].fillna('').astype(str)                    # empty cells stay empty (no 'nan' text)
            for _i, _note in MD_NOTES.items():
                _mask = _m['compound_id'].astype(str) == _i
                _m.loc[_mask, 'notes'] = (_m.loc[_mask, 'notes'].str.strip() + ' | ' + _note).str.strip(' |')
            plan.append((_m.to_csv(index=False).encode(), f'{dd}/{f.name}', 'MD summary (+ notes)'))
        else:
            plan.append((f, f'{dd}/{f.name}', 'MD report'))

# --- redocking overlays ---
if INCLUDE_OVERLAYS:
    for f in sorted((RUN / 'validation').glob('*_overlay_complex.pdb')): plan.append((f, f'data/incoming/validation/{f.name}', 'overlay'))

# --- the shared contract package, if it is in Drive and not yet in the repo ---
cdir = PROJECT / 'omniphyto_contract'
if cdir.is_dir() and not (CLONE / 'omniphyto_contract' / 'schemas.py').exists():
    for f in sorted(cdir.glob('*.py')): plan.append((f, f'omniphyto_contract/{f.name}', 'contract package'))

# --- everything else produced on Drive: prepared inputs, validation files, PLIP, notebooks (kept OUT of data/incoming, which the app reads) ---
if INCLUDE_PREPARED:
    for nm in ('dpp4', 'herg', 'cyp3a4', 'cyp2d6'):
        d0 = RUN / 'receptors'
        for suffix in ('.pdbqt', '.box.pdb', '.box.txt', '_H.pdb'): add_file(d0 / f'{nm}{suffix}', f'data/prepared/receptors/{nm}{suffix}', '(prepared receptor)')
        cf = d0 / f'{nm}_vina.conf'
        if cf.exists():          # make the config portable: the receptor path becomes the bare file name
            txt_ = re.sub(r'(?m)^receptor\s*=.*$', f'receptor = {nm}.pdbqt', cf.read_text())
            plan.append((txt_.encode(), f'data/prepared/receptors/{nm}_vina.conf', '(Vina config, portable path)'))
        else: plan.append((None, f'data/prepared/receptors/{nm}_vina.conf', 'MISSING in Drive'))
    for f in sorted((RUN / 'receptors').glob('ligand_*_crystal.pdb')): plan.append((f, f'data/prepared/receptors/{f.name}', '(crystal ligand)'))
    for f in sorted((RUN / 'ligands').glob('*.pdbqt')): plan.append((f, f'data/prepared/ligands/{f.name}', '(ligand input)'))
    for f in sorted((RUN / 'validation').glob('*')):
        if f.is_file() and 'overlay' not in f.name and f.suffix in ('.csv', '.pdbqt'): plan.append((f, f'data/validation/{f.name}', '(redocking file)'))
if INCLUDE_PLIP:
    for f in sorted((RUN / 'results' / 'plip').glob('*_complex.pdb')): plan.append((f, f'data/plip/{f.name}', '(PLIP complex)'))
if INCLUDE_NOTEBOOKS:
    nbdir = Path(os.environ.get('OMNI_NOTEBOOKS', '/content/drive/MyDrive/Colab Notebooks'))
    for f in sorted(nbdir.glob('*.ipynb')) if nbdir.is_dir() else []:
        if re.search(r'omni|herba', f.name, re.I):
            try:
                raw_ = f.read_text(encoding='utf-8')
                if re.search(r'ghp_[A-Za-z0-9]{20,}|github_pat_[A-Za-z0-9_]{20,}', raw_):
                    print(f'SKIPPED notebook {f.name}: it contains something that looks like a GitHub token. Remove it from the notebook on Drive and revoke that token.'); continue
                nbj = json.loads(raw_)
                for c in nbj.get('cells', []):
                    if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
                plan.append((json.dumps(nbj, indent=1).encode(), f'notebooks/{f.name}', '(notebook, outputs removed)'))
            except Exception: pass

# --- skeleton: folders, ignore rules, drop-folder readme ---
GITIGNORE_ADD = ['__pycache__/', '*.pyc', '.venv/', '.env', '.streamlit/secrets.toml', 'data/output/', '.DS_Store']
INCOMING_README = """# data/incoming: the drop folder

Copy result files here and refresh the app. File names are matched by pattern; the newest file wins.

| Pattern | What it is |
|---|---|
| docking_target*.csv | DPP-4 docking, all docked compounds |
| docking_antitargets*.csv | safety docking (hERG, CYP3A4, CYP2D6) |
| summary_report*.csv | top-compound report (scores, residues, redocking verdicts) |
| redocking*.csv | redocking RMSD per protein |
| *scored_detail*.csv, *scored.csv | ML engineer tables |
| *compounds_with_DOIs*.csv | master compound table (SMILES, DOIs) |
| ranking*.csv | final ML ranking (arrives later) |
| target_card.json | disease, target, PDB, reference ligand, safety panel |
| results/<target|herg|cyp3a4|cyp2d6>/poses/*.pdbqt | pose files named in the pose_file column |
| md_report/ | GROMACS outputs (later) |

Receptor structures (3D4L.pdb, 8ZYO.pdb, 2V0M.pdb, 3TDA.pdb) go in data/receptors/.
Delivery is by git: the sync notebook commits files here, and the hosted app redeploys with them.
"""
MD_README = """# data/incoming/md_report: MD results (the app's MD page reads this folder)

Put here, when the MD run is finished:

- `md_summary.csv` with the columns: compound_id, label (candidate|control), length_ns, ligand_rmsd_mean_A, ligand_rmsd_sd_A, protein_rmsd_mean_A, hbond_occupancy_pct, notes
- plots named `<compound_id>_rmsd.png`, `<compound_id>_rmsf.png`, `<compound_id>_hbonds.png`

`candidate` = the compound; `control` = the crystal drug (ligand 605) simulated the same way. The MD page lights up when these files appear.
"""
skeleton = {'data/incoming/README.md': INCOMING_README.encode(), 'data/receptors/.gitkeep': b'', 'data/poses/.gitkeep': b'',
            'data/incoming/md_report/README.md': MD_README.encode(), 'assets/.gitkeep': b'', 'engines/__init__.py': b'"""Adapters for a future docking / MD engine. Pages only read files; see README_FIRST section 8."""\n'}
for rel, content in skeleton.items():
    if not (CLONE / rel).exists(): plan.append((content, rel, 'new skeleton file'))

# --- manifest of everything in the plan (what is where) ---
def _grp(rel):
    parts = rel.split('/'); return '/'.join(parts[:3]) if rel.startswith('data/incoming/results/') else '/'.join(parts[:2]) if len(parts) > 2 else rel
_counts = {}
for src_, rel_, note_ in plan:
    if src_ is not None: _counts[_grp(rel_)] = _counts.get(_grp(rel_), 0) + 1
MANIFEST = ['# OmniPhyto data manifest', f'_generated {datetime.now():%Y-%m-%d %H:%M} by the Drive-to-GitHub notebook_', '', '## Delivered (files per folder)', '']
MANIFEST += [f'- `{k}`: {v} files' for k, v in sorted(_counts.items())]
MANIFEST += ['', '## Expected, not delivered yet', '',
             '- `data/incoming/ranking.csv`: full ranking of all docked compounds from the ML engineer (only the top 10 so far)',
             '- `data/incoming/md_report/`: MD results (delivered if `omniphyto_run/md_report/` exists on Drive; see the README in that folder)',
             '- poses for compounds docked later: copy the new `results/` folder into `data/incoming/`', '',
             '## Where things are', '',
             '- `data/incoming/`: read by the app (tables, poses, target card)', '- `data/receptors/`: raw RCSB structures for the 3D viewer',
             '- `data/prepared/`: receptors and ligands as docked (PDBQT, box, config) for reproducibility',
             '- `data/validation/`: redocking inputs, poses and RMSD tables', '- `data/plip/`: interaction complexes for the top compounds', '- `notebooks/`: the notebooks that produced the results']
plan.append(('\n'.join(MANIFEST).encode() + b'\n', 'data/MANIFEST.md', 'manifest'))

# --- show the plan ---
rows, total, problems = [], 0, []
for src, rel, note in plan:
    dest = CLONE / rel
    if src is None: rows.append(('MISSING', rel, '-', note)); problems.append(rel); continue
    data = src if isinstance(src, bytes) else None
    size = len(data) if data is not None else src.stat().st_size
    if dest.exists():
        same = (dest.read_bytes() == data) if data is not None else (hashlib.md5(dest.read_bytes()).hexdigest() == hashlib.md5(src.read_bytes()).hexdigest())
        state = 'unchanged' if same else 'UPDATE'
    else: state = 'new'
    if size > 95e6: state = 'TOO BIG (>95 MB, skipped)'
    elif size > 50e6: state += ' (large!)'
    rows.append((state, rel, human(size), note)); total += size if state in ('new', 'UPDATE') else 0
summary = pd.DataFrame(rows, columns=['state', 'destination in repo', 'size', 'note'])
non_pose = summary[~summary.note.eq('pose')]
print(non_pose.to_string(index=False))
ps = summary[summary.note.eq('pose')]
print(f'\n+ {len(ps)} pose files ({ps.state.value_counts().to_dict()})')
print(f'\nTo be written: {human(total)} | missing items: {len(problems)}')
for p in problems: print('  MISSING:', p)

def ignored_by_gitignore(rels):
    r = subprocess.run(['git', 'check-ignore', '--no-index', '-v', '--stdin'], input='\n'.join(rels), cwd=CLONE, capture_output=True, text=True)
    hits = {}
    for line in r.stdout.splitlines():
        if '\t' not in line: continue
        rule, path = line.split('\t', 1); pat = rule.split(':', 2)[2]
        if not pat.startswith('!'): hits[path] = pat
    return hits
PLANNED = [rel for src, rel, note in plan if src is not None and not (len(src) if isinstance(src, bytes) else src.stat().st_size) > 95e6]
IGN = ignored_by_gitignore(PLANNED)
if IGN:
    by_rule = {}
    for pth, pat in IGN.items(): by_rule.setdefault(pat, []).append(pth)
    print(f'\nWARNING: your .gitignore would silently skip {len(IGN)} of the planned files:')
    for pat, lst in by_rule.items(): print(f'  rule "{pat}" skips {len(lst)} files, e.g. {lst[0]}')
    print('  (the notebook adds exceptions for them automatically when you copy for real)')
STALE = [p for p in git('ls-files').splitlines() if p == 'data/md_report/.gitkeep']      # folder from an earlier layout
PYC = [p for p in git('ls-files').splitlines() if '__pycache__/' in p or p.endswith('.pyc')] + STALE
REAL_PYC = [x for x in PYC if x not in STALE]          # genuine compiled files only (STALE is handled separately)
if REAL_PYC: print(f'\n{len(REAL_PYC)} compiled Python files (.pyc) are tracked in git; they will be removed from git (your files on disk are not touched).')
if STALE: print(f'\n{len(STALE)} old placeholder file(s) from an earlier layout will be removed: {STALE}')
if DRY_RUN:
    print('\nDRY RUN: nothing copied. Set DRY_RUN = False in Cell 1 and run Cells 1, 2 (optional), 3 again to copy into the clone.')
else:
    written = []
    for src, rel, note in plan:
        if src is None: continue
        dest = CLONE / rel
        size = len(src) if isinstance(src, bytes) else src.stat().st_size
        if size > 95e6: continue
        dest.parent.mkdir(parents=True, exist_ok=True)
        if isinstance(src, bytes): dest.write_bytes(src)
        else: shutil.copy2(src, dest)
        written.append(rel)
    if STALE: git('rm', '-q', '-f', '--ignore-unmatch', *STALE)        # old placeholder: remove from git AND disk, or the next 'git add -A' would bring it back
    if REAL_PYC: git('rm', '-r', '--cached', '-q', '--ignore-unmatch', *REAL_PYC)      # only when there really are .pyc files (an empty list makes git fail)
    gi = CLONE / '.gitignore'; cur = gi.read_text().splitlines() if gi.exists() else []
    add = [x for x in GITIGNORE_ADD if x not in cur]
    if add: gi.write_text('\n'.join(cur + ([''] if cur and cur[-1] else []) + ['# added by the OmniPhyto sync notebook'] + add) + '\n')
    if IGN:                                                       # re-include ignored folders (every parent folder down to the file)
        neg = []
        for pth in IGN:
            parts = pth.split('/')[:-1]
            for i in range(1, len(parts) + 1):
                d_ = '/'.join(parts[:i])
                for line in (f'!/{d_}/',): 
                    if line not in neg: neg.append(line)
            g_ = f'!/{"/".join(parts)}/**'
            if g_ not in neg: neg.append(g_)
        cur2 = gi.read_text().splitlines()
        gi.write_text('\n'.join(cur2 + ['', '# exceptions so result files are not ignored (added by the sync notebook)'] + [n for n in neg if n not in cur2]) + '\n')
        left = ignored_by_gitignore(written)
        print('Ignore exceptions added. Files still ignored:', len(left))
        if left: raise SystemExit('Some files are still ignored by .gitignore: ' + ', '.join(list(left)[:5]))
    PLANNED_RELS = written
    print(f'\nCopied {len(written)} files into the local clone (not committed yet).')
    st_ = git('status', '--short').splitlines(); print('\n'.join(st_[:8]) + ('\n...' if len(st_) > 8 else ''))

In [ ]:
# CELL 4: COMMIT and PUSH (needs PUSH = True and DRY_RUN = False)
if DRY_RUN: raise SystemExit('Set DRY_RUN = False in Cell 1 and run Cell 3 first, so the files are in the clone.')
status_ = git('status', '--porcelain')
if not status_: raise SystemExit('Nothing to commit: the repo already has these files.')
git('checkout', '-B', PUSH_BRANCH)
git('add', '-A')
tracked_ = set(git('ls-files').splitlines())
missing_ = [r for r in globals().get('PLANNED_RELS', []) if r not in tracked_]
if missing_:
    raise SystemExit(f'STOP: {len(missing_)} planned files are not tracked by git (nothing was pushed), e.g. {missing_[:3]}. Re-run Cell 3 with DRY_RUN = False.')
print(f'Check passed: all {len(globals().get("PLANNED_RELS", []))} copied files are tracked by git.')
msg = f'Add OmniPhyto results and data layout ({datetime.now():%Y-%m-%d %H:%M})'
git('commit', '-m', msg)
print(git('log', '-1', '--stat', '--pretty=%h %s').splitlines()[0])
print(f'{len(status_.splitlines())} changed paths committed on branch {PUSH_BRANCH}')
if not PUSH:
    print('\nCommitted locally only. Set PUSH = True in Cell 1 (and re-run Cells 1, 3, 4) to push.')
else:
    remote_sha = git('ls-remote', AUTH_URL, f'refs/heads/{PUSH_BRANCH}').split()[:1]        # this branch is owned by the notebook: replace it, but only if nobody else changed it meanwhile
    lease = [f'--force-with-lease={PUSH_BRANCH}:{remote_sha[0]}'] if remote_sha else []
    print(git('push', *lease, AUTH_URL, f'{PUSH_BRANCH}:{PUSH_BRANCH}'))
    print(f'\nPushed to {CLEAN_URL} branch {PUSH_BRANCH}.')
    if PUSH_BRANCH != BRANCH: print(f'Next: open a pull request {PUSH_BRANCH} -> {BRANCH} on GitHub, or point the hosting service at branch {PUSH_BRANCH}.')
    else: print('The hosted site redeploys from this branch automatically (if it is connected to GitHub).')